In [1]:
# Import libries
import numpy as np
import pandas as pd
import os
import re
import glob
import json
import csv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyranges as pr
import pysam
# from scipy.stats import mannwhitneyu, stats
# from statannotations.Annotator import Annotator

current_directory = os.getcwd()
print("Current Directory:", current_directory)
pd.set_option("display.max_columns", None)


Current Directory: /mnt/NAS3/home/jiwon/ECTRES/python


## AmpliconArchitect

In [6]:
aaSuite_gemline_ms=pd.read_csv('../summary/aaSuite_germline_ms/10X/aaSuite_gemline_ms_all_20260327.csv')
aaSuite_gemline_ms[['source_barcode','sample_id']].drop_duplicates()['source_barcode'].value_counts()

source_barcode
ECGI1    34
H2170    32
EFM19    11
Name: count, dtype: int64

In [8]:
aaSuite_gemline_ms.head(2)

,amplicon_barcode,aa_barcode,amplicon_number,amplicon_decomposition_class,ecDNA+,BFB+,ecDNA_amplicons,AmpliconID,amplicon_index,aa_summary_file_path,N_Intervals,Intervals,OncogenesAmplified,TotalIntervalSize,AmplifiedIntervalSize,AverageAmplifiedCopyCount,N_Chromosomes,N_SequenceEdges,N_BreakpointEdges,N_CoverageShifts,N_MeanshiftSegmentsCopyCount>5,N_Foldbacks,N_CoverageShiftsWithBreakpointEdges,source_barcode,sample_id,amplicon_type,old_source_barcode,old_sample_id
0,ECTRES-EFM19-0001-TPX-A05-WGS-4GXSY5-amplicon10,ECTRES-EFM19-0001-TPX-A05-WGS-4GXSY5,amplicon10,No amp/Invalid,None detected,None detected,0,10,amplicon10,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,15:20824661-20979641,",",154981,154980,2.750840,1,1,0,0,0,0,0,EFM19,EFM_5,none,EFM19,EFM_5
1,ECTRES-EFM19-0001-TPX-A05-WGS-4GXSY5-amplicon11,ECTRES-EFM19-0001-TPX-A05-WGS-4GXSY5,amplicon11,Linear,None detected,None detected,0,11,amplicon11,/mnt/NAS3/home/jiwon/HL-NF/scratch/ECTRES/resu...,1,16:33268118-33533123,",",265006,265003,3.730855,1,3,0,2,0,0,2,EFM19,EFM_5,Linear,EFM19,EFM_5


In [9]:
cna_seg=pd.read_csv('../summary/ichorCNA/ECTRES_ichorCNA_cna_seg_20260331.csv')

cna_seg.shape

(402633, 17)

In [10]:
## ECDNA STructure 분석

In [28]:
import pandas as pd
import re
import math
from collections import defaultdict

source = 'H2170'
df = aaSuite_gemline_ms[(aaSuite_gemline_ms['source_barcode']==source)&(aaSuite_gemline_ms['amplicon_type']!='none')]

# 확인
print(df.shape)
print(df.columns.tolist())


# =========================
# 1. interval parsing 함수
# =========================
def parse_intervals(interval_str):
    """
    '3:111-222,8:333-444' -> list of dicts
    """
    out = []
    if pd.isna(interval_str):
        return out

    for chunk in str(interval_str).split(","):
        chunk = chunk.strip()
        m = re.match(r'^(?:chr)?([^:]+):(\d+)-(\d+)$', chunk)
        if not m:
            continue

        chrom, start, end = m.groups()
        start, end = int(start), int(end)

        if start > end:
            start, end = end, start

        out.append({
            "chrom": str(chrom),
            "start": start,
            "end": end,
            "raw": f"{chrom}:{start}-{end}"
        })
    return out


def interval_len(x):
    return x["end"] - x["start"] + 1


def overlap_len(a, b):
    if a["chrom"] != b["chrom"]:
        return 0
    return max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]) + 1)


def reciprocal_overlap(a, b):
    ov = overlap_len(a, b)
    if ov == 0:
        return 0.0
    return min(ov / interval_len(a), ov / interval_len(b))


def endpoint_distance(a, b):
    """
    start/end 차이 중 큰 값
    """
    if a["chrom"] != b["chrom"]:
        return math.inf
    return max(abs(a["start"] - b["start"]), abs(a["end"] - b["end"]))


def is_close_interval(a, b, recip_thr=0.80, endpoint_tol=200_000, containment_slack=200_000):
    """
    '엄청 가까운' 기준
    1) reciprocal overlap >= 0.8
    또는
    2) 양 끝 경계 차이가 endpoint_tol 이내
    또는
    3) 거의 containment 관계
    """
    if a["chrom"] != b["chrom"]:
        return False

    ro = reciprocal_overlap(a, b)
    if ro >= recip_thr:
        return True

    if endpoint_distance(a, b) <= endpoint_tol:
        return True

    # a가 b 안에 거의 포함 / b가 a 안에 거의 포함
    if (a["start"] >= b["start"] - containment_slack) and (a["end"] <= b["end"] + containment_slack):
        return True

    if (b["start"] >= a["start"] - containment_slack) and (b["end"] <= a["end"] + containment_slack):
        return True

    return False


def interval_score(a, b):
    """
    close candidate가 여러 개일 때 우선순위
    reciprocal overlap 높을수록 좋고,
    endpoint distance 작을수록 좋음
    """
    ro = reciprocal_overlap(a, b)
    dist = endpoint_distance(a, b)
    return (ro, -dist)


# =========================
# 2. parental row만 추출
# =========================
parental = df[df["sample_id"].eq("parental")].copy()

# parental amplicon 정렬용 숫자
parental["amp_num_numeric"] = (
    parental["amplicon_number"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(int)
)

# amplicon_type + amplicon_number 기준 정렬
parental = parental.sort_values(["amplicon_type", "amp_num_numeric"]).copy()

print("\n[Parental amplicons]")
print(parental[["amplicon_barcode", "amplicon_type", "Intervals", "OncogenesAmplified"]].to_string(index=False))


# =========================
# 3. parental interval label 생성
# =========================
# 예:
# ecDNA1_A, ecDNA1_B ...
# ecDNA2_A ...
# BFB1_A ...
# CNC1_A ...
# Linear1_A ...

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

parental_type_counter = defaultdict(int)
parental_segments = []

for _, row in parental.iterrows():
    amp_type = row["amplicon_type"]
    parental_type_counter[amp_type] += 1
    amp_idx = parental_type_counter[amp_type]

    intervals = parse_intervals(row["Intervals"])

    for i, iv in enumerate(intervals):
        label = f"{amp_type}{amp_idx}_{LETTERS[i]}"

        parental_segments.append({
            **iv,
            "label": label,
            "parent_amplicon_type": amp_type,
            "parent_amplicon_index": amp_idx,
            "parent_amplicon_barcode": row["amplicon_barcode"],
            "parent_genes": row["OncogenesAmplified"]
        })

parental_label_df = pd.DataFrame(parental_segments)

print("\n[Parental segment labels]")
print(parental_label_df[["label", "raw", "parent_amplicon_type", "parent_genes"]].to_string(index=False))


# =========================
# 4. clone interval annotation
# =========================
# 원칙:
# (1) parental exact match -> 그대로 label
# (2) parental near match  -> label + "'"
# (3) unmatched -> novel label 생성
# (4) 이후 다른 clone에서 비슷한 novel이 나오면 그 label 재사용

novel_catalog = []
novel_counters = defaultdict(int)

annot_rows = []

for _, row in df.iterrows():
    sample_id = row["sample_id"]
    amp_barcode = row["amplicon_barcode"]
    amp_type = row["amplicon_type"]
    genes = row["OncogenesAmplified"]

    intervals = parse_intervals(row["Intervals"])

    for iv in intervals:
        # ---------------------
        # 4-1. parental 우선 검색
        # ---------------------
        parental_candidates = parental_label_df[parental_label_df["chrom"] == iv["chrom"]].to_dict("records")

        exact_parental = [
            p for p in parental_candidates
            if (p["start"] == iv["start"]) and (p["end"] == iv["end"])
        ]

        if exact_parental:
            best = exact_parental[0]
            assigned_label = best["label"]
            match_status = "exact_parental"
            matched_source = "parental"
            matched_interval_raw = best["raw"]

        else:
            close_parental = [
                p for p in parental_candidates
                if is_close_interval(iv, p)
            ]

            if close_parental:
                best = sorted(close_parental, key=lambda x: interval_score(iv, x), reverse=True)[0]
                assigned_label = best["label"] + "'"
                match_status = "near_parental"
                matched_source = "parental"
                matched_interval_raw = best["raw"]

            else:
                # ---------------------
                # 4-2. novel catalog 검색
                # ---------------------
                novel_candidates = [
                    n for n in novel_catalog
                    if (n["amplicon_type"] == amp_type) and (n["chrom"] == iv["chrom"])
                ]

                exact_novel = [
                    n for n in novel_candidates
                    if (n["start"] == iv["start"]) and (n["end"] == iv["end"])
                ]

                if exact_novel:
                    best = exact_novel[0]
                    assigned_label = best["label"]
                    match_status = "exact_novel_reused"
                    matched_source = "novel_catalog"
                    matched_interval_raw = best["raw"]

                else:
                    close_novel = [
                        n for n in novel_candidates
                        if is_close_interval(iv, n)
                    ]

                    if close_novel:
                        best = sorted(close_novel, key=lambda x: interval_score(iv, x), reverse=True)[0]
                        assigned_label = best["label"] + "'"
                        match_status = "near_novel_reused"
                        matched_source = "novel_catalog"
                        matched_interval_raw = best["raw"]

                    else:
                        # ---------------------
                        # 4-3. 완전 신규 label 생성
                        # ---------------------
                        novel_counters[amp_type] += 1
                        assigned_label = f"{amp_type}_{novel_counters[amp_type]}"
                        match_status = "novel_created"
                        matched_source = "new"
                        matched_interval_raw = None

                        novel_catalog.append({
                            **iv,
                            "label": assigned_label,
                            "amplicon_type": amp_type
                        })

        annot_rows.append({
            "sample_id": sample_id,
            "amplicon_barcode": amp_barcode,
            "amplicon_type": amp_type,
            "interval_raw": iv["raw"],
            "assigned_label": assigned_label,
            "match_status": match_status,
            "matched_source": matched_source,
            "matched_interval_raw": matched_interval_raw,
            "genes": genes,
            "AverageAmplifiedCopyCount": row["AverageAmplifiedCopyCount"]

        })

annot_df = pd.DataFrame(annot_rows)

print("\n[Annotation result head]")
print(annot_df.head(20).to_string(index=False))

print("\n[match_status counts]")
print(annot_df["match_status"].value_counts())

novel_df = pd.DataFrame(novel_catalog)
print("\n[Novel catalog]")
if len(novel_df) > 0:
    print(novel_df.to_string(index=False))
else:
    print("No novel intervals created.")


# =========================
# 5. clone별로 보기 좋게 정리
# =========================
# 각 amplicon마다 label들을 묶어서 보는 표
amplicon_summary = (
    annot_df
    .groupby(["sample_id", "amplicon_barcode", "amplicon_type", "AverageAmplifiedCopyCount"], as_index=False)
    .agg(
        assigned_labels=("assigned_label", lambda x: ", ".join(x)),
        raw_intervals=("interval_raw", lambda x: ", ".join(x)),
        match_statuses=("match_status", lambda x: ", ".join(x))
    )
)

print("\n[Amplicon summary head]")
print(amplicon_summary.head(20).to_string(index=False))


# =========================
# 6. sample x label 매트릭스 (눈으로 보기 좋음)
# =========================
# 각 sample에 어떤 label이 있는지 1/0 매트릭스
sample_label_matrix = (
    annot_df.assign(value=1)
    .pivot_table(
        index="sample_id",
        columns="assigned_label",
        values="value",
        aggfunc="max",
        fill_value=0
    )
    .sort_index()
)

print("\n[Sample x Label matrix shape]")
print(sample_label_matrix.shape)

display(sample_label_matrix)


# =========================
# 7. 저장
# # =========================
# parental_label_df.to_csv("H2170_parental_segment_labels.csv", index=False)
# annot_df.to_csv("H2170_clone_interval_annotation.csv", index=False)
# amplicon_summary.to_csv("H2170_amplicon_summary_by_clone.csv", index=False)
# sample_label_matrix.to_csv("H2170_sample_by_label_matrix.csv")

print("\nSaved:")
print("- H2170_parental_segment_labels.csv")
print("- H2170_clone_interval_annotation.csv")
print("- H2170_amplicon_summary_by_clone.csv")
print("- H2170_sample_by_label_matrix.csv")

(112, 28)
['amplicon_barcode', 'aa_barcode', 'amplicon_number', 'amplicon_decomposition_class', 'ecDNA+', 'BFB+', 'ecDNA_amplicons', 'AmpliconID', 'amplicon_index', 'aa_summary_file_path', 'N_Intervals', 'Intervals', 'OncogenesAmplified', 'TotalIntervalSize', 'AmplifiedIntervalSize', 'AverageAmplifiedCopyCount', 'N_Chromosomes', 'N_SequenceEdges', 'N_BreakpointEdges', 'N_CoverageShifts', 'N_MeanshiftSegmentsCopyCount>5', 'N_Foldbacks', 'N_CoverageShiftsWithBreakpointEdges', 'source_barcode', 'sample_id', 'amplicon_type', 'old_source_barcode', 'old_sample_id']

[Parental amplicons]
                               amplicon_barcode amplicon_type                                                                                                   Intervals                                       OncogenesAmplified
 ECTRES-H2170-0001-TPX-A01-WGS-3YV111-amplicon2           BFB                                                                                       1:114642776-115572702                

assigned_label,BFB1_A,BFB1_A',CNC_1,Linear_1,Linear_1',Linear_2,Linear_2',Linear_3,ecDNA1_A,ecDNA1_A',ecDNA1_B,ecDNA1_B',ecDNA1_C,ecDNA1_D,ecDNA1_E,ecDNA2_A,ecDNA2_B,ecDNA2_C,ecDNA2_C',ecDNA_1
sample_id,,,,,,,,,,,,,,,,,,,,
NCI_1,1,0,0,0,1,0,0,0,0,1,0,1,1,1,1,0,0,0,1,0
NCI_11,1,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,1,0,0
NCI_12,1,0,0,0,1,0,0,0,0,1,0,1,1,0,1,0,0,0,1,0
NCI_13,1,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,1,0
NCI_14,1,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,1,0,0
NCI_15,1,0,0,0,0,1,0,0,0,1,0,1,1,0,1,0,0,1,0,0
NCI_16,1,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,1,0
NCI_17,1,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,1,0
NCI_18,1,0,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,1,0



Saved:
- H2170_parental_segment_labels.csv
- H2170_clone_interval_annotation.csv
- H2170_amplicon_summary_by_clone.csv
- H2170_sample_by_label_matrix.csv


In [33]:
amplicon_summary.to_csv('../summary/aaSuite_germline_ms/10X/amplicon_summary_annotation_seg_NCI.csv',index=False)